# NotebookLM Kit — Flashcard Pipeline

Generates one flashcard set per source, downloads all as JSON.

1. **Setup** — authenticate and verify `tsx`
2. **Config** — set your notebook ID
3. **Template** — tune card count, difficulty, and optional focus prompt
4. **List Sources** — preview sources; pick indices to filter
5. **Create** — submit one flashcard job per source; jobs saved to disk
6. **Poll** — wait for generation, or resume from a previous run
7. **Download** — save each set as `<timestamp>_<Notebook>__<Source>.json`


In [ ]:
import importlib, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from pipeline import load_credentials, check_tsx, login, SDK_ROOT

# First time only — opens a browser window for Google login, saves credentials.json:
#   login()

creds = load_credentials(mode="patchright")
check_tsx()

Credentials ready — token: 42 chars, cookies: 2316 chars
tsx: tsx v4.22.3
node v20.17.0


## Config

Find your notebook ID in the NotebookLM URL:  
`https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**

In [ ]:
NOTEBOOK_ID = "64d7d18f-a2db-4b0b-959d-5276506dffad"  # ← paste your notebook ID here

## List Sources

`list_sources(notebook_id, creds)` — prints an indexed table of all sources (title, status, full ID).  

In [ ]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)



Notebook : Mastering Modern Data Orchestration with Dagster
+---+--------------------------------------------------------------------------+------------+--------------------------------------+
| # | Title                                                                    | Status     | Source ID                            |
+---+--------------------------------------------------------------------------+------------+--------------------------------------+
| 0 | A Quick Introduction to Machine Learning with Dagster | HackerNoon       | READY      | fbcb2f70-5f48-4a30-93ce-6099e82a0cd5 |
| 1 | Best Practices for Machine Learning with Dagster                         | READY      | e06a6437-95a0-4bb0-b886-66fcb4e3b57a |
| 2 | Dagster for Data Orchestration: Architecture & Capabilities - Atlan      | READY      | 7b94d750-50a8-4d6c-a059-9fdc8d2bd502 |
| 3 | Dagster training: Orchestrating data pipelines in the cloud - Ambient IT | READY      | bf05939f-1b17-44f5-a64e-6a2a83e5d514 |
| 4 | Da

## Flashcard Template

Optionally provide a focus prompt and adjust card count / difficulty.

In [ ]:
# Optional: leave empty to let NotebookLM decide focus automatically
FLASHCARD_INSTRUCTIONS = "Focus on key definitions and any formulas. Avoid trivia."

FLASHCARD_CUSTOMIZATION = {
    "numberOfCards": 2,    # 1 = Fewer cards,  2 = Standard / More cards
    "difficulty":    3,    # 1 = Easy,          2 = Medium,              3 = Hard
    # "language": "en",    # BCP-47 language code — omit to use notebook default
}

## Create Flashcard Sets

Job IDs are saved to `jobs/flashcards/<timestamp>_jobs.json` — safe to re-run the poll/download cells without re-submitting.

In [ ]:
from pipeline import create_artifacts

fc_jobs = create_artifacts(
    NOTEBOOK_ID, "FLASHCARDS", sources[1],
    FLASHCARD_CUSTOMIZATION, FLASHCARD_INSTRUCTIONS,
    creds,
)

# sources = sources[:2]                   # first two
# sources = [sources[i] for i in [0, 2]]  # specific picks


Submitted 1 job(s)  →  jobs\flashcards\20260530_103338_Mastering_Modern_Data_Orchestr.json
+---+--------------------------------------------------+----------------------------------------------+
| # | Source                                           | Artifact ID                                  |
+---+--------------------------------------------------+----------------------------------------------+
| 0 | Best Practices for Machine Learning with Dagster | 46537b9b-bf0d-4521-8573-cbe5d1a1ddf2         |
+---+--------------------------------------------------+----------------------------------------------+


## Poll Until Ready

Run **the cell below** to wait automatically.  
Or run the **resume cell** first if you're continuing a previous session (jobs already submitted).

In [ ]:
# ── Resume cell — run this instead of Create if you already submitted jobs ───
from pipeline import load_jobs
fc_jobs = load_jobs("FLASHCARDS")


Loaded 1 job(s) from 20260530_103338_Mastering_Modern_Data_Orchestr.json
+---+--------------------------------------------------+----------------------------------------------+
| # | Source                                           | Artifact ID                                  |
+---+--------------------------------------------------+----------------------------------------------+
| 0 | Best Practices for Machine Learning with Dagster | 46537b9b-bf0d-4521-8573-cbe5d1a1ddf2         |
+---+--------------------------------------------------+----------------------------------------------+


In [ ]:
from pipeline import poll_jobs

poll_jobs(fc_jobs, NOTEBOOK_ID, creds, interval=15, max_wait_min=15)

  ✓ Best Practices for Machine Learning with Dagster        READY

✅ All artifacts ready — proceed to download.


True

## Download

Each set is saved as a JSON file in `outputs/flashcards/`.

In [ ]:
from pipeline import download_artifacts

download_artifacts(fc_jobs, NOTEBOOK_ID, "FLASHCARDS", creds)

ℹ 1 job(s) missing notebookTitle/createdAt — fetching from SDK to backfill…
  ✓ Jobs file updated: 20260530_103338_Mastering_Modern_Data_Orchestr.json

Downloaded 1 / 1 artifact(s)  →  d:\Core\_Code D\notebooklm-kit\outputs\flashcards
+---+--------------------------------------------------+----------------------------------------------+------------+
| # | Source                                           | File                                         | Status     |
+---+--------------------------------------------------+----------------------------------------------+------------+
| 0 | Best Practices for Machine Learning with Dagster | 20260530_103336_Mastering_Modern_Data_Orc... | downloaded |
+---+--------------------------------------------------+----------------------------------------------+------------+


[{'sourceTitle': 'Best Practices for Machine Learning with Dagster',
  'notebookTitle': 'Mastering Modern Data Orchestration with Dagster',
  'createdAt': '2026-05-30T05:03:36.048Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\flashcards\\20260530_103336_Mastering_Modern_Data_Orchestr__Best_Practices_for_Machine_Lea__FLASHCARDS.json',
  'status': 'downloaded'}]